# Semantic Embedding Search

Build semantic search index using sentence embeddings. Uses FAISS for efficient nearest-neighbor lookup.

## Output Files
- **src/semantic_retriever.py** - SemanticRetriever class (auto-generated)
- **data/processed/semantic_index/faiss_index** - FAISS index file

## Workflow
1. Check prerequisites (corpus.pkl exists)
2. Install sentence-transformers and FAISS
3. Auto-generate/update src/semantic_retriever.py
4. Load enriched corpus
5. Encode documents to embeddings
6. Build and save FAISS index
7. Test search functionality (verify it works)

## 1. Setup and Prerequisites Check

Initializes paths and verifies that corpus.pkl from Phase 1 exists.

**Output:** Project root path and prerequisite file status

In [1]:
from pathlib import Path
import sys
import os

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

corpus_file = project_root / "data" / "processed" / "corpus.pkl"
semantic_index_dir = project_root / "data" / "processed" / "semantic_index"
semantic_index_file = semantic_index_dir / "faiss_index"

print(f"Project root: {project_root}")
print()
print("Prerequisites check:")
print(f"  corpus.pkl exists: {corpus_file.exists()}")

if not corpus_file.exists():
    print("\nERROR: Run 02_data_preparation.ipynb first to generate corpus.pkl")
else:
    print("\nAll prerequisites ready.")

Project root: /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki

Prerequisites check:
  corpus.pkl exists: True

All prerequisites ready.


## 2. Install Dependencies

Installs sentence-transformers (embeddings) and faiss-cpu (vector search).

**Output:** Installation status for each library

In [2]:
import subprocess
import sys

print("Checking dependencies...\n")

dependencies = [("sentence-transformers", "sentence_transformers"), ("faiss-cpu", "faiss")]

for package_name, import_name in dependencies:
    try:
        __import__(import_name)
        print(f"  {package_name}: already installed")
    except ImportError:
        print(f"  {package_name}: installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
        print(f"  {package_name}: done")

print("\nAll dependencies ready.")

Checking dependencies...

  sentence-transformers: already installed
  faiss-cpu: already installed

All dependencies ready.


## 3. Auto-Generate/Update src/semantic_retriever.py

Creates or updates SemanticRetriever class for encoding and searching. Writes to src/semantic_retriever.py.

**Output:** File path, creation/update status, and file size

In [3]:
semantic_retriever_code = '''
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Tuple

class SemanticRetriever:
    """Semantic search using FAISS and sentence embeddings."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.corpus = None
    
    def build_index(self, corpus: List[str]) -> None:
        """Encode corpus and build FAISS index."""
        self.corpus = corpus
        print(f"Encoding {len(corpus)} documents...")
        embeddings = self.model.encode(corpus, show_progress_bar=True, batch_size=128)
        embeddings = embeddings.astype('float32')
        
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)
        print(f"Index built: {self.index.ntotal} vectors (dim={embeddings.shape[1]})")
    
    def search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Search for similar documents."""
        if self.index is None:
            raise ValueError("Index not built. Call build_index() first.")
        
        query_embedding = self.model.encode([query], show_progress_bar=False).astype('float32')
        distances, indices = self.index.search(query_embedding, top_k)
        return [(int(idx), float(dist)) for idx, dist in zip(indices[0], distances[0])]
    
    def save(self, path: str) -> None:
        """Save index to disk."""
        faiss.write_index(self.index, str(path))
        print(f"Index saved to {path}")
    
    def load(self, path: str) -> None:
        """Load index from disk."""
        self.index = faiss.read_index(str(path))
        print(f"Index loaded from {path}")
'''

src_dir = project_root / "src"
src_dir.mkdir(exist_ok=True)
semantic_retriever_path = src_dir / "semantic_retriever.py"

# Check if file exists
file_existed = semantic_retriever_path.exists()

# Write the file (creates if not exists, updates if exists)
with open(semantic_retriever_path, "w") as f:
    f.write(semantic_retriever_code)

file_size = semantic_retriever_path.stat().st_size
status = "updated" if file_existed else "created"

print(f"File {status}: src/semantic_retriever.py")
print(f"Path: {semantic_retriever_path}")
print(f"Size: {file_size} bytes")

File updated: src/semantic_retriever.py
Path: /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/src/semantic_retriever.py
Size: 1700 bytes


## 4. Load Enriched Corpus

Loads the enriched corpus from Phase 1 (contains book title, rating, review, etc).

**Output:** Number of documents loaded and sample preview

In [4]:
import pickle

print("Loading enriched corpus...")

with open(corpus_file, "rb") as f:
    corpus = pickle.load(f)

print(f"Loaded: {len(corpus):,} enriched documents")
print(f"\nSample document:")
print("-" * 70)
print(corpus[0][:300] + "...")
print("-" * 70)

Loading enriched corpus...
Loaded: 110,167 enriched documents

Sample document:
----------------------------------------------------------------------
Book: The Slow-Cooker Savant: Become a Dinner Hero 52 Weeks of the Year!
Product Rating: 5.0/5 | Review: 5.0/5 | Verified: Yes
Very nice book<br />Great recipes...
----------------------------------------------------------------------


## 5. Build Semantic Index

Encodes all documents using all-MiniLM-L6-v2 (384 dimensions) and builds FAISS L2 index. Takes 2-5 minutes.

**Output:** Index statistics and save location

In [5]:
# Force reimport to get latest version
import importlib
if 'src.semantic_retriever' in sys.modules:
    importlib.reload(sys.modules['src.semantic_retriever'])

from src.semantic_retriever import SemanticRetriever

print("Building semantic index...")
print("(This takes 2-5 minutes)\n")

semantic_retriever = SemanticRetriever()
semantic_retriever.build_index(corpus)

print(f"\nSaving index...")
semantic_index_dir.mkdir(parents=True, exist_ok=True)
semantic_retriever.save(str(semantic_index_file))
print(f"Saved to: {semantic_index_file}")

Building semantic index...
(This takes 2-5 minutes)

Encoding 110167 documents...


Batches:   0%|          | 0/861 [00:00<?, ?it/s]

Index built: 110167 vectors (dim=384)

Saving index...
Index saved to /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
Saved to: /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index


## 6. Verify Semantic Index Works

Tests that generated SemanticRetriever class actually works by running search queries and verifying results.

**Output:** Actual search results showing top 3 matches with distances and book titles

In [6]:
import pandas as pd

print("Testing semantic search functionality...\n")

# Load books data for display
books_data = pd.read_parquet(project_root / "data" / "processed" / "books_sample.parquet")

test_queries = ["python programming", "mystery thriller", "machine learning"]

print("Semantic Search Results")
print("=" * 80)

all_passed = True

for query in test_queries:
    try:
        results = semantic_retriever.search(query, top_k=3)
        print(f"\nQuery: '{query}'")
        
        if len(results) != 3:
            print(f"  ERROR: Expected 3 results, got {len(results)}")
            all_passed = False
            continue
        
        for rank, (doc_id, distance) in enumerate(results, 1):
            # Verify document ID is valid
            if doc_id < 0 or doc_id >= len(corpus):
                print(f"  ERROR: Invalid doc_id {doc_id}")
                all_passed = False
                continue
            
            book_title = books_data.iloc[doc_id]['product_title']
            review_rating = books_data.iloc[doc_id]['rating']
            print(f"  {rank}. Doc #{doc_id} - {book_title} ({review_rating}/5) | Distance: {distance:.4f}")
    
    except Exception as e:
        print(f"  ERROR: {str(e)}")
        all_passed = False

print("\n" + "=" * 80)

if all_passed:
    print("VERIFICATION PASSED: Semantic search working correctly")
    print("Index ready for use in 06_rag.ipynb")
else:
    print("VERIFICATION FAILED: Check errors above")

Testing semantic search functionality...

Semantic Search Results

Query: 'python programming'
  1. Doc #94234 - Automate the Boring Stuff with Python: Practical Programming for Total Beginners (5.0/5) | Distance: 0.4746
  2. Doc #90252 - Automate the Boring Stuff with Python: Practical Programming for Total Beginners (5.0/5) | Distance: 0.5999
  3. Doc #212 - Python Programming for the Absolute Beginner, 3rd Edition (5.0/5) | Distance: 0.6061

Query: 'mystery thriller'
  1. Doc #16004 - I See You (4.0/5) | Distance: 0.6031
  2. Doc #3075 - The Suspect: A Thriller (5.0/5) | Distance: 0.7515
  3. Doc #83545 - Hide and Seek: A gripping fast-paced psychological thriller with a shocking twist (4.0/5) | Distance: 0.7563

Query: 'machine learning'
  1. Doc #20475 - Data Mining: Practical Machine Learning Tools and Techniques (The Morgan Kaufmann Series in Data Management Systems) (5.0/5) | Distance: 1.0108
  2. Doc #104728 - Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow:

## 7. Load and Test Saved Index

Tests that saved index can be loaded and used correctly in fresh instance.

**Output:** Index load confirmation and verification of loaded index functionality

In [7]:
print("Testing index persistence (load from disk)...\n")

# Create fresh instance and load from disk
test_retriever = SemanticRetriever()
test_retriever.load(str(semantic_index_file))
test_retriever.corpus = corpus

# Test a query with loaded index
test_query = "good book recommendation"
results = test_retriever.search(test_query, top_k=2)

print(f"Query: '{test_query}'")
for rank, (doc_id, distance) in enumerate(results, 1):
    book_title = books_data.iloc[doc_id]['product_title']
    print(f"  {rank}. {book_title} (distance: {distance:.4f})")

print("\nPersistence verified: Index saves and loads correctly.")

Testing index persistence (load from disk)...

Index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
Query: 'good book recommendation'
  1. The Last Lecture (distance: 0.7500)
  2. Follow You Home (distance: 0.7602)

Persistence verified: Index saves and loads correctly.
